# Predição de Compatibilidade Auditor–Projeto com ML Ensemble
### Notebook explicativo — do dataset à matriz de scores

Este notebook acompanha, célula a célula, todo o pipeline do artigo:

1. Carregamento e inspeção dos dados sintéticos
2. Entendimento do *fit score* (variável-alvo)
3. Treinamento e comparação de cinco modelos *ensemble*
4. Análise de importância de *features*
5. Construção da matriz de scores
6. Heurísticas de alocação: Greedy vs. Húngaro

> **Reprodutibilidade:** todas as etapas usam `random_state=42`. Rodar de cima a baixo reproduz exatamente os números do artigo.


## 0. Dependências

Instale as bibliotecas (rode uma vez):

In [ ]:
# !pip install pandas numpy scikit-learn xgboost lightgbm catboost scipy matplotlib
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ambiente pronto.")

## 1. Carregando os dados

Três arquivos CSV compõem o dataset sintético:

| Arquivo | Conteúdo | Dimensão |
|---|---|---|
| `auditores.csv` | Perfil dos auditores | 200 × 47 |
| `projetos.csv` | Requisitos dos projetos | 300 × 48 |
| `alocacoes.csv` | Pares históricos (treino do ML) | 1.500 × 84 |

O `alocacoes.csv` é o que alimenta o modelo: cada linha é um par auditor×projeto com o `fit_score` já calculado, que o modelo aprende a prever.

In [ ]:
from pathlib import Path
import os

# Caminho dos dados: usa a pasta ../data relativa ao notebook,
# ou a variável de ambiente AUDITORES_PROJECT_DATA se definida.
DATA_DIR = Path(os.environ.get("AUDITORES_PROJECT_DATA", Path.cwd().parent / "data"))
print("Lendo dados de:", DATA_DIR)

auditores = pd.read_csv(DATA_DIR / "auditores.csv")
projetos  = pd.read_csv(DATA_DIR / "projetos.csv")
alocacoes = pd.read_csv(DATA_DIR / "alocacoes.csv")

print("auditores:", auditores.shape)
print("projetos :", projetos.shape)
print("alocacoes:", alocacoes.shape)
alocacoes[["auditor_id","projeto_id","skills_match_pct","nivel_match","fit_score"]].head()

## 2. A variável-alvo: *fit score*

O `fit_score` mede a compatibilidade entre um auditor e um projeto, de 0 a 1. É uma média ponderada de seis dimensões:

$$fit = 0{,}40\cdot skills + 0{,}20\cdot nivel + 0{,}15\cdot setor + 0{,}15\cdot cert + 0{,}05\cdot idioma + 0{,}05\cdot dispon.$$

Vamos olhar sua distribuição.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(alocacoes["fit_score"], bins=40, color="#2563EB", alpha=0.8)
ax[0].set_title("Distribuição do fit_score"); ax[0].set_xlabel("fit_score")
ax[1].scatter(alocacoes["skills_match_pct"], alocacoes["fit_score"], s=8, alpha=0.3, color="#16A34A")
ax[1].set_title("fit_score vs. skills_match_pct"); ax[1].set_xlabel("skills_match_pct"); ax[1].set_ylabel("fit_score")
plt.tight_layout(); plt.show()
print("Média:", round(alocacoes.fit_score.mean(),3), "| Desvio:", round(alocacoes.fit_score.std(),3))

## 3. Preparando treino e teste

Separamos as *features* (X) do alvo (y) e dividimos em 80% treino / 20% teste. As colunas de identificação são descartadas — não carregam sinal preditivo.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold

TARGET = "fit_score"
EXCLUIR = ["alocacao_id","auditor_id","projeto_id",TARGET]
FEATURES = [c for c in alocacoes.columns if c not in EXCLUIR]

X, y = alocacoes[FEATURES], alocacoes[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]} | Features: {len(FEATURES)}")

## 4. Treinando os cinco modelos *ensemble*

Comparamos duas famílias:
- **Bagging** (árvores independentes, média): Random Forest, Extra Trees
- **Boosting** (árvores sequenciais, correção de erros): XGBoost, LightGBM, CatBoost

Para cada modelo medimos R², MAE, RMSE no teste e a validação cruzada de 5 dobras.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

modelos = {
    "Random Forest": RandomForestRegressor(n_estimators=300, max_features="sqrt", min_samples_leaf=2, n_jobs=-1, random_state=42),
    "XGBoost":  XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31, subsample=0.8, random_state=42, verbose=-1),
    "CatBoost": CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6, random_state=42, verbose=0),
    "Extra Trees": ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
}

resultados, ajustados = {}, {}
for nome, m in modelos.items():
    cv = cross_val_score(m, X_train, y_train, cv=kf, scoring="r2", n_jobs=-1)
    m.fit(X_train, y_train)
    yp = np.clip(m.predict(X_test), 0, 1)
    resultados[nome] = dict(R2=r2_score(y_test,yp), CV=cv.mean(), CV_std=cv.std(),
                            MAE=mean_absolute_error(y_test,yp), RMSE=np.sqrt(mean_squared_error(y_test,yp)))
    ajustados[nome] = m

pd.DataFrame(resultados).T.round(4)

**Leitura dos resultados:** o CatBoost lidera (R²≈0,974), com XGBoost e LightGBM logo atrás. O Random Forest é o mais fraco — confirma o padrão da literatura de que *boosting* supera *bagging* em dados tabulares heterogêneos.

## 5. Importância das *features*

Quais variáveis o melhor modelo mais usa para prever? Isso dá interpretabilidade ao sistema.

In [ ]:
melhor = max(resultados, key=lambda k: resultados[k]["R2"])
mb = ajustados[melhor]
print("Melhor modelo:", melhor)

fi = pd.DataFrame({"feature": FEATURES, "importance": mb.feature_importances_})
fi = fi.sort_values("importance", ascending=False).head(10)

plt.figure(figsize=(9,4))
plt.barh(fi["feature"][::-1], fi["importance"][::-1], color="#2563EB", alpha=0.85)
plt.title(f"Top 10 features — {melhor}"); plt.xlabel("Importância"); plt.tight_layout(); plt.show()
fi

## 6. Construindo a matriz de scores

Agora aplicamos o conceito ao uso prático: calcular o fit de cada auditor para cada projeto futuro. A função abaixo replica a fórmula do fit score (Seção 3.2).

In [ ]:
def fit_score(aud, proj):
    s_aud=set(json.loads(aud["skills"]));         s_req=set(json.loads(proj["skills_requeridas"]))
    c_aud=set(json.loads(aud["certificacoes"]));  c_req=set(json.loads(proj["certs_requeridas"]))
    se_aud=set(json.loads(aud["setores_experiencia"]))
    sm  = len(s_aud & s_req)/max(len(s_req),1)
    gap = aud["nivel_num"]-proj["nivel_min_num"]
    nm  = max(0.,1.+gap*0.4) if gap<0 else min(1.,1.-gap*0.05)
    setm= 1.0 if proj["setor_cliente"] in se_aud else 0.0
    cm  = len(c_aud & c_req)/max(len(c_req),1)
    idi = float(aud["ingles"]) if proj["ingles_requerido"] else 1.0
    disp= aud["disponibilidade_pct"]/100.
    return round(float(np.clip(.4*sm+.2*nm+.15*setm+.15*cm+.05*idi+.05*disp,0,1)),3)

aud5, proj5 = auditores.head(5), projetos.head(5)
M = np.array([[fit_score(a,p) for _,p in proj5.iterrows()] for _,a in aud5.iterrows()])
pd.DataFrame(M, index=aud5["auditor_id"], columns=proj5["projeto_id"])

## 7. Heurísticas de alocação: Greedy vs. Húngaro

A matriz dá os scores, mas cada auditor só pode ir a um projeto. Duas estratégias:

- **Greedy:** para cada projeto, pega o melhor disponível (ótimo local)
- **Húngaro:** maximiza a soma total de scores (ótimo global)

In [ ]:
from scipy.optimize import linear_sum_assignment

def greedy(M):
    al, res = set(), {}
    for j in range(M.shape[1]):
        col = M[:,j].copy(); col[list(al)] = -1
        b = int(np.argmax(col)); res[j]=b; al.add(b)
    return res

def hungaro(M):
    r,c = linear_sum_assignment(-M)
    return {int(cc):int(rr) for rr,cc in zip(r,c)}

g, h = greedy(M), hungaro(M)
soma_g = sum(M[g[j],j] for j in range(5))
soma_h = sum(M[h[j],j] for j in range(5))
print(f"Greedy  — soma: {soma_g:.3f}")
print(f"Húngaro — soma: {soma_h:.3f}")

### Quando cada heurística vence?

O resultado depende da razão **auditores disponíveis / projetos**:

- **Abundância** (200 auditores, 10 projetos): Greedy se sai bem — há candidatos de sobra.
- **Escassez** (matriz quadrada, nº auditores ≈ nº projetos): Húngaro vence — a otimização global evita desperdiçar bons candidatos.

A célula abaixo demonstra isso variando a razão.

In [ ]:
for n_proj, n_aud in [(5,200),(10,200),(50,60),(50,50)]:
    ps = projetos.sample(n_proj, random_state=42).reset_index(drop=True)
    aus = auditores.sample(n_aud, random_state=42).reset_index(drop=True)
    MM = np.array([[fit_score(a,p) for _,p in ps.iterrows()] for _,a in aus.iterrows()])
    gg, hh = greedy(MM), hungaro(MM)
    sg = sum(MM[gg[j],j] for j in range(n_proj))
    sh = sum(MM[hh[j],j] for j in list(hh)[:n_proj])
    venc = "Húngaro" if sh>sg else "Greedy"
    print(f"{n_aud:>3} aud / {n_proj:>2} proj  ->  Greedy={sg:6.2f}  Hungaro={sh:6.2f}  vence: {venc}")

## Conclusão

Este notebook reproduziu todo o pipeline do artigo:

✅ Carregamento dos dados sintéticos
✅ Treinamento de 5 modelos ensemble (CatBoost foi o melhor)
✅ Análise de importância de features
✅ Construção da matriz de scores
✅ Comparação de heurísticas de alocação

O framework é diretamente aplicável a dados reais: basta substituir os CSVs sintéticos por dados da firma, mantendo a mesma estrutura de colunas.